In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
CY_Data = pd.read_csv(r"CY_Data.csv",encoding='latin1', low_memory=False)

In [ ]:
## Next Quarter Product Sales Forecasting using Machine Learning

CY_Data['TRANSACTION_DATE'] = pd.to_datetime(CY_Data['TRANSACTION_DATE'])
CY_Data['Quarter'] = CY_Data['TRANSACTION_DATE'].dt.to_period('Q')
quarter_sales = CY_Data.groupby(['Quarter','PRODUCT_NAME'])['Value'].sum().reset_index()
quarter_sales['Quarter_Num'] = quarter_sales['Quarter'].astype(str).str.extract('(\d)').astype(int)


forecast = []
for product in quarter_sales['PRODUCT_NAME'].unique():
    df = quarter_sales[quarter_sales['PRODUCT_NAME'] == product]
    X = df[['Quarter_Num']]
    y = df['Value']
    if len(df) > 1:
        model = LinearRegression()
        model.fit(X,y)
        next_quarter = df['Quarter_Num'].max() + 1
        pred = model.predict([[next_quarter]])
        forecast.append({
            "Product": product,
            "Next_Quarter_Sales": pred[0]
        })
Sale_forecast = pd.DataFrame(forecast)

print(Sale_forecast)

In [ ]:
## Product Sales Forecasting and Slab Optimization using Machine Learning

CY_Data['TRANSACTION_DATE'] = pd.to_datetime(CY_Data['TRANSACTION_DATE'])
CY_Data['Quarter'] = CY_Data['TRANSACTION_DATE'].dt.to_period('Q')
quarter_sales = CY_Data.groupby(['Quarter','PRODUCT_NAME'])['Volume_in_Ltr'].sum().reset_index()
quarter_sales['Quarter_Num'] = quarter_sales['Quarter'].factorize()[0] + 1

results = []
for product in quarter_sales['PRODUCT_NAME'].unique():
    df = quarter_sales[quarter_sales['PRODUCT_NAME'] == product]
    if len(df) > 1:
        X = df[['Quarter_Num']]
        y = df['Volume_in_Ltr']
        model = LinearRegression()
        model.fit(X,y)
        next_q = df['Quarter_Num'].max() + 1
        pred_sales = model.predict([[next_q]])[0]
        suggested_slab = pred_sales * 0.7
        results.append({
            "Product": product,
            "Predicted_Q4_Sales": pred_sales,
            "Suggested_Slab": suggested_slab
        })

Slab_forecast = pd.DataFrame(results)
print(Slab_forecast)

In [ ]:
## DBT (Direct Benefit Transfer) Prediction based on Product Demand Forecast

CY_Data['TRANSACTION_DATE'] = pd.to_datetime(CY_Data['TRANSACTION_DATE'])
CY_Data['Quarter'] = CY_Data['TRANSACTION_DATE'].dt.to_period('Q')
quarter_sales = CY_Data.groupby(['Quarter','PRODUCT_NAME'])['Volume_in_Ltr'].sum().reset_index()
quarter_sales['Quarter_Num'] = quarter_sales['Quarter'].factorize()[0] + 1

results = []
for product in quarter_sales['PRODUCT_NAME'].unique():
    df = quarter_sales[quarter_sales['PRODUCT_NAME'] == product]
    if len(df) > 1:
        X = df[['Quarter_Num']]
        y = df['Volume_in_Ltr']
        model = LinearRegression()
        model.fit(X,y)
        next_q = df['Quarter_Num'].max() + 1
        pred_volume = model.predict([[next_q]])[0]
        dbt_rate = 5   
        predicted_dbt = pred_volume * dbt_rate
        results.append({
            "Product": product,
            "Predicted_Q4_Volume": pred_volume,
            "Estimated_DBT": predicted_dbt
        })

dbt_forecast = pd.DataFrame(results)
print(dbt_forecast)